In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
# Write your code here
from torchvision.transforms import transforms
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import ImageFolder
import os
import random
import torch
import matplotlib.pyplot as plt

train_dir = os.path.join(path, 'PlantVillage', 'train')
test_dir = os.path.join(path, 'PlantVillage', 'test')


transform = transforms.Compose([
    transforms.Resize((32,32)),
    transforms.RandomRotation(15),
    transforms.ToTensor()
])

train_dataset = ImageFolder(root=train_dir, transform=transform)
test_dataset = ImageFolder(root=test_dir, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

# Create a function to visualize samples
def visualize_samples(dataset, num_samples=8, title="Dataset Samples"):
    """
    Visualize random samples from a dataset.

    Args:
        dataset: PyTorch Dataset object
        num_samples: Number of samples to display
        title: Title for the plot
    """
    # Select random indices
    indices = random.sample(range(len(dataset)), num_samples)

    # Calculate grid size
    cols = 4
    rows = (num_samples + cols - 1) // cols

    # Create the plot
    fig, axes = plt.subplots(rows, cols, figsize=(12, 3 * rows))
    axes = axes.flatten()

    for i, idx in enumerate(indices):
        # Get image and label
        image, label = dataset[idx]
        # Convert tensor to numpy for display
        if isinstance(image, torch.Tensor):
            image = image.permute(1, 2, 0).numpy()  # CHW -> HWC
        # Get class name
        class_name = dataset.classes[label]
        # Display image
        axes[i].imshow(image)
        axes[i].set_title(f"{class_name}\n(Label: {label})", fontsize=10)
        axes[i].axis('off')
    # Hide any unused subplots
    for i in range(num_samples, len(axes)):
        axes[i].axis('off')

    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


visualize_samples(train_dataset, 8)

In [ ]:
# Write your code here
import torch.nn as nn
class PotatoModel(nn.Module):

  def __init__(self, input_ch, output_ch):
    super().__init__()

    self.features = nn.Sequential(
        nn.Conv2d(input_ch, 16, kernel_size=3, stride=1, padding=1), # 16x32x32 (32-3+2+1)
        nn.BatchNorm2d(16),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=2, stride=2), #16x16x16

        nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1), # 32x16x16
        nn.BatchNorm2d(32),
        nn.MaxPool2d(kernel_size=2, stride=2),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=2, stride=2), # 32x8x8

        nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1), # 64x8x8
        nn.BatchNorm2d(64),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=2, stride=2), #64x4x4

        nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1), # 128x4x4
        nn.BatchNorm2d(128),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=2, stride=2), # 128x2x2

        nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1), # 256x2x2
        nn.BatchNorm2d(256),
        nn.ReLU(inplace=True)
    )

    self.classifier = nn.Sequential(
        nn.Linear(256*1*1, 128),
        nn.ReLU(inplace=True),

        nn.Linear(128, 256),
        nn.ReLU(inplace=True),
        nn.Linear(256, output_ch)
    )

  def forward(self, x):

    x = self.features(x)
    x = torch.flatten(x, 1)
    x = self.classifier(x)

    return x

In [ ]:
from tqdm import tqdm    # Shows progress bar

# 🔹 Training Loop
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()  # Set model to training mode
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in tqdm(dataloader):
        images, labels = images.to(device), labels.to(device)


        outputs = model(images)  # Forward pass
        loss = criterion(outputs, labels)  # Compute loss

        optimizer.zero_grad()  # Reset gradients
        loss.backward()  # Backpropagation
        optimizer.step()  # Update weights

        total_loss += loss.item()

        # Track accuracy
        outputs = torch.softmax(outputs, dim=1)
        predictions = outputs.argmax(dim=1)  # Get class with highest probability
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total  # Compute accuracy in percentage
    return avg_loss, accuracy

# 🔹 Validation Loop
def validate(model, dataloader, criterion, device):
    model.eval()  # Set model to evaluation mode
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():  # Disable gradient computation
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)  # Forward pass
            loss = criterion(outputs, labels)  # Compute loss
            total_loss += loss.item()

            # Compute accuracy
            outputs = torch.softmax(outputs, dim=1)
            predictions = outputs.argmax(dim=1)  # Get predicted class
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total  # Compute accuracy in percentage
    return avg_loss, accuracy


In [ ]:
# Write your code here
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

num_epochs = 10

model = PotatoModel(3, 3)
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

for epoch in range(num_epochs):
    train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_accuracy = validate(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    print(f"Epoch {epoch+1}/{num_epochs}: "
          f"Train Loss={train_loss:.4f}, Train Accuracy={train_accuracy:.2f}%, "
          f"Val Loss={val_loss:.4f}, Val Accuracy={val_accuracy:.2f}%")


import matplotlib.pyplot as plt

# Plot loss curve
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs+1), train_losses, label="Train Loss", marker='o')
plt.plot(range(1, num_epochs+1), val_losses, label="Validation Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()

# Plot accuracy curve
plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs+1), train_accuracies, label="Train Accuracy", marker='o')
plt.plot(range(1, num_epochs+1), val_accuracies, label="Validation Accuracy", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy Curve")
plt.legend()

plt.show()


In [ ]:
# Write your code here
# Write your code here
import torch.nn as nn
class PotatoModel(nn.Module):

  def __init__(self, input_ch, output_ch):
    super().__init__()

    self.conv1 = nn.Conv2d(input_ch, 16, kernel_size=3, stride=1, padding=1) # 16x32x32 (32-3+2+1)
    self.b1 = nn.BatchNorm2d(16)
    self.relu = nn.ReLU(inplace=True)
    self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2) #16x16x16

    self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1) # 32x16x16
    self.b2 = nn.BatchNorm2d(32)

    self.conv3 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1) # 64x8x8
    self.b3 = torch._nnpack_available.BatchNorm2d(64)

    self.conv4 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1) # 128x4x4
    self.b4 = nn.BatchNorm2d(128)
    self.conv5 = nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1) # 256x2x2
    self.conv6 = nn.BatchNorm2d(256)

    self.classifier = nn.Sequential(
        nn.Linear(256*1*1, 128),


        nn.Linear(128, 256),

        nn.Linear(256, output_ch)
    )

  def forward(self, x):

    conv1 = self.maxpool(self.relu(self.b(self.conv1(x))))
    conv2 = self.maxpool(self.relu(self.b(self.conv1(conv1))))
    conv2 = torch.cat([x, conv2], dim=1)
    conv3 = self.maxpool(self.relu(self.b(self.conv1(conv2))))
    conv3 = torch.cat([conv1, conv3], dim=1)
    conv4 = self.maxpool(self.relu(self.b(self.conv1(conv3))))
    conv4 = torch.cat([conv2, conv4], dim=1)
    conv5 = self.maxpool(self.relu(self.b(self.conv1(conv4))))

    x = torch.flatten(256*2*2, 1)

    x = self.classifier(x)


    return x